# Module 13: Training Fundamentals


## 🧠 Welcome to Part 5: Training & Fine-Tuning

Up until now, we have used spaCy's pre-trained models (like `en_core_web_sm`) or we have built hard-coded rules (like the `Matcher`). 

But what if you want the model to automatically recognize a completely new entity type, like `DRUG_NAME` or `FLIGHT_NUMBER`, even in sentences it has never seen before? 

You need to **train** a statistical model. In spaCy v3+, the training process was completely overhauled to be incredibly robust, using a unified configuration system and a highly efficient binary data format.


<br><br>

---

<br><br>


## 📏 Rules vs. Training

When should you write a Rule (Part 3) and when should you Train a model (Part 5)?

| | Write a Rule (Matcher) | Train a Model (Machine Learning) |
|---|---|---|
| **Use Case** | Finite lists (e.g. all countries), exact patterns (e.g. `###-###-####`), standard terminology. | Infinite possibilities (e.g. people's names), ambiguous context ("Apple" the fruit vs company). |
| **Data Needed** | A dictionary of terms or regex patterns. | Hundreds or thousands of labeled example sentences. |
| **Accuracy** | 100% precision on exact matches. Fails if the word is slightly misspelled or phrasing changes. | Can generalize to unseen words and misspellings based on context. Can make weird mistakes. |


<br><br>

---

<br><br>


## 📦 The `Example` Object and `DocBin`

To train a model, you have to show it pairs of data: the text, and the correct answer (the "Gold Standard"). 
In spaCy, this pair is represented by the `Example` object.

Once you have hundreds of `Example` objects, you save them to disk using a `DocBin`, which compresses the data into a highly efficient `.spacy` binary file.


In [ ]:
import spacy
from spacy.training import Example
from spacy.tokens import DocBin

nlp = spacy.blank("en")

# Let's pretend we have a raw sentence
text = "I bought a new iPhone 14 yesterday."
predicted_doc = nlp.make_doc(text) # The raw text processed into a blank doc

# We create a reference document that has the correct answers manually labeled
reference_doc = nlp.make_doc(text)
# Let's say "iPhone 14" starts at token 4 and ends at token 6
reference_doc.ents = [reference_doc.char_span(15, 24, label="GADGET")]

# Combine them into an Example
example = Example(predicted_doc, reference_doc)
print(f"Example created: {example}")

# Save it to a DocBin (we will do this at scale in the next module)
db = DocBin()
db.add(reference_doc)
db.to_disk("train.spacy")
print("Saved to train.spacy!")


TypeError: object of type 'NoneType' has no len()

<br><br>

---

<br><br>


## ⚙️ The Config System (`config.cfg`)

In older versions of spaCy, training required writing massive, complex Python scripts with nested loops. 

In spaCy v3+, training is driven by a configuration file: `config.cfg`. This file defines everything: the learning rate, the batch size, the neural network architecture, and the pipeline components.

You almost never write a `config.cfg` from scratch. You generate a base config using the command line!


In [ ]:
# Run this command in your terminal to generate a base config for an NER model:
# !python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency

print("Run the command above in your terminal (or un-comment it) to generate the config file!")


<br><br>

---

<br><br>


## 🚀 The Training Loop

Once you have your data saved as `train.spacy` and `dev.spacy` (for evaluation), and your `config.cfg` ready, you start the training process entirely from the command line.

You **DO NOT** write Python loops for training in spaCy v3.

```bash
# The master training command:
python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy
```


<br><br>

---

<br><br>


## 📈 Understanding the Output

When training runs, spaCy prints a table showing the progress. Here is what you need to look out for:

- **LOSS**: The error rate of the model. This should go down over time. If it spikes or goes to `NaN`, your learning rate is too high or your data is corrupt.
- **E** (Epochs): How many times the model has seen the entire dataset.
- **#** (Steps): The number of individual training batches processed.
- **SCORE (F-Score)**: The F1-Score of your model on the `dev.spacy` (testing) data. This should go UP over time.

### Early Stopping
spaCy watches the SCORE. If the score doesn't improve for a certain number of steps (called "patience"), spaCy will automatically stop training to prevent **overfitting** (where the model memorizes the training data but fails on real-world data).

## 🎉 Summary of Module 13

You now understand the architecture of modern spaCy training:
1. Prepare data by creating `Example` objects and saving them into binary `DocBin` (`.spacy`) files.
2. Generate a `config.cfg` file using the CLI.
3. Run `python -m spacy train` from the command line.

In **Module 14: Training NER Models**, we will write a complete script to convert a custom dataset into `.spacy` format and run a real training loop to teach the AI a brand new entity type!
